# 06 — Full Credit Paper

## Purpose
End-to-end single-borrower analysis demonstrating what a credit analyst presents to the investment committee. This notebook ties together all previous modules into a complete credit assessment.

**This is what a credit analyst does every day:**
1. Borrower overview (industry, size, structure)
2. Financial summary (spread + normalised)
3. Key ratio dashboard with trends
4. Working capital assessment
5. Repayment capacity analysis
6. Internal rating / scorecard result
7. PD estimation (Z-score + Merton)
8. Proposed pricing
9. Recommended covenants
10. Auto-generated committee narrative

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_sme_dataset
from src.financial_spreading import spread_borrower, format_spread_display
from src.normalisation import normalise_earnings, total_available_for_servicing
from src.ratio_engine import calculate_ratios, borrower_ratio_summary, BANK_THRESHOLDS
from src.trend_analysis import analyse_trends
from src.working_capital import borrower_wc_analysis, analyse_working_capital
from src.altman_zscore import borrower_zscore_trend
from src.merton_pd import borrower_merton_analysis
from src.credit_scorecard import score_borrower
from src.risk_based_pricing import compute_pricing, pricing_waterfall_table
from src.commentary_generator import generate_borrower_commentary, generate_automated_comments_table

pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

# Load and prepare data
df = generate_sme_dataset(n_borrowers=80, seed=42)
df_r = calculate_ratios(df)

# Select the base case borrower
BID = 0
borrower_name = df[df['borrower_id']==BID]['borrower_name'].iloc[0]
industry = df[df['borrower_id']==BID]['anzsic_division'].iloc[0]
print(f'Credit Assessment: {borrower_name}')
print(f'Industry: {industry}')

---

## 1. Financial Summary

In [ ]:
spread = spread_borrower(df, BID)
display_spread = format_spread_display(spread, borrower_name)

print(f'\n{"="*70}')
print(f'SECTION 1: FINANCIAL STATEMENT SPREAD')
print(f'{"="*70}')
display(display_spread)

## 2. Earnings Normalisation

In [ ]:
waterfall = normalise_earnings(spread)

print(f'\n{"="*70}')
print(f'SECTION 2: EARNINGS NORMALISATION (EBIT → EBITDAO)')
print(f'{"="*70}')

wf_display = waterfall.copy()
for col in ['FY-2', 'FY-1', 'FY0']:
    wf_display[col] = wf_display[col].apply(lambda x: f'${x:,.0f}' if pd.notna(x) else '')
display(wf_display)

## 3. Key Ratios & Trend Analysis

In [ ]:
trends = analyse_trends(df_r, BID)

print(f'\n{"="*70}')
print(f'SECTION 3: THREE-PERIOD TREND REVIEW')
print(f'Core flags passed: {trends["pass"].sum()} / {len(trends)}')
print(f'{"="*70}')

def colour_status(val):
    if val == 'POSITIVE': return 'background-color: #C8E6C9'
    elif val == 'NEGATIVE': return 'background-color: #FFCDD2'
    return 'background-color: #FFF9C4'

display(
    trends[['metric', 'FY-2', 'FY-1', 'FY0', 'status', 'comment']]
    .style.applymap(colour_status, subset=['status'])
)

In [ ]:
# Key ratio charts for the credit paper
ref = df_r[df_r['borrower_id'] == BID].sort_values('period')
periods = ['FY-2', 'FY-1', 'FY0']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Row 1: Revenue, EBITDA, ICR, DSCR
axes[0,0].bar(periods, ref['revenue']/1e6, color='#2196F3', alpha=0.8)
axes[0,0].set_title('Revenue (A$M)', fontweight='bold')

axes[0,1].bar(periods, ref['ebitda']/1e6, color='#4CAF50', alpha=0.8)
axes[0,1].set_title('EBITDA (A$M)', fontweight='bold')

axes[0,2].plot(periods, ref['icr'], 'o-', linewidth=2, color='#FF9800', markersize=8)
axes[0,2].axhline(2.0, color='red', linestyle='--', alpha=0.6)
axes[0,2].set_title('ICR', fontweight='bold')

axes[0,3].plot(periods, ref['dscr'], 'o-', linewidth=2, color='#9C27B0', markersize=8)
axes[0,3].axhline(1.2, color='red', linestyle='--', alpha=0.6)
axes[0,3].set_title('DSCR', fontweight='bold')

# Row 2: Debt/EBITDA, Current Ratio, FCF, Net Worth
axes[1,0].plot(periods, ref['debt_to_ebitda'], 'o-', linewidth=2, color='#F44336', markersize=8)
axes[1,0].axhline(4.0, color='red', linestyle='--', alpha=0.6)
axes[1,0].set_title('Debt / EBITDA', fontweight='bold')

axes[1,1].plot(periods, ref['current_ratio'], 'o-', linewidth=2, color='#00BCD4', markersize=8)
axes[1,1].axhline(1.2, color='red', linestyle='--', alpha=0.6)
axes[1,1].set_title('Current Ratio', fontweight='bold')

axes[1,2].bar(periods, ref['free_cash_flow']/1e6, color='#8BC34A', alpha=0.8)
axes[1,2].axhline(0, color='red', linestyle='--', alpha=0.6)
axes[1,2].set_title('Free Cash Flow (A$M)', fontweight='bold')

axes[1,3].bar(periods, ref['net_worth']/1e6, color='#3F51B5', alpha=0.8)
axes[1,3].set_title('Net Worth (A$M)', fontweight='bold')

fig.suptitle(f'Credit Dashboard — {borrower_name}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/06_credit_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Working Capital Assessment

In [ ]:
wc_table = borrower_wc_analysis(df, BID)
fy0 = df_r[(df_r['borrower_id']==BID) & (df_r['period']=='FY0')].iloc[0]
wc = analyse_working_capital(fy0)

print(f'\n{"="*70}')
print(f'SECTION 4: WORKING CAPITAL ANALYSIS')
print(f'Overall Flag: {wc["flag"]} — {wc["comment"]}')
print(f'{"="*70}')
display(wc_table)

## 5. Credit Quality — Z-Score & Merton PD

In [ ]:
zt = borrower_zscore_trend(df, BID)
merton = borrower_merton_analysis(df, BID)

print(f'\n{"="*70}')
print(f'SECTION 5: CREDIT QUALITY ASSESSMENT')
print(f'{"="*70}')
print(f'\nAltman Z-Score:')
display(zt)
print(f'Trend: {zt.attrs.get("trend", "N/A")} — {zt.attrs.get("trend_comment", "")}')

print(f'\nMerton PD Model:')
print(f'  PD: {merton["pd"]:.4%} ({merton["pd_comment"]})')
print(f'  Distance-to-Default: {merton["dd"]:.3f}')
print(f'  PVEL: ${merton["pvel"]:,.0f} ({merton["pvel_comment"]})')
print(f'  Implied LGD: {merton["implied_lgd"]:.2%}')

## 6. Scorecard & Committee Decision

In [ ]:
sc = score_borrower(df_r, BID)

print(f'\n{"="*70}')
print(f'SECTION 6: INTEGRATED CREDIT SCORECARD')
print(f'Weighted Score: {sc["weighted_score"]:.2%} | Grade: {sc["grade"]} | Decision: {sc["decision"]}')
print(f'{"="*70}')

sc_table = sc['scorecard_table'].copy()
sc_table['actual_fmt'] = sc_table['actual'].apply(
    lambda x: f'{x:.4f}' if isinstance(x, (int, float)) and not pd.isna(x) else str(x)
)

def colour_pass(val):
    return 'background-color: #C8E6C9' if val == 1 else 'background-color: #FFCDD2'

display(
    sc_table[['metric', 'actual_fmt', 'rule', 'pass', 'weight']]
    .rename(columns={'actual_fmt': 'actual'})
    .style.applymap(colour_pass, subset=['pass'])
)

## 7. Risk-Based Pricing

In [ ]:
pricing = compute_pricing(
    facility_amount=fy0['total_debt'],
    pd_value=merton['pd'],
    pvel=merton['pvel'],
    weighted_score=sc['weighted_score'],
    debt_to_ebitda=fy0['debt_to_ebitda'],
    wc_flag=wc['flag'],
    icr=fy0['icr'],
    dscr=fy0['dscr'],
)

waterfall = pricing_waterfall_table(pricing)

print(f'\n{"="*70}')
print(f'SECTION 7: RISK-BASED PRICING')
print(f'Indicative All-in Rate: {pricing["all_in_rate"]:.2%}')
print(f'{"="*70}')

wf = waterfall.copy()
wf['Rate'] = wf['Rate'].apply(lambda x: f'{x:.4%}')
display(wf)

## 8. Automated Committee Narrative

In [ ]:
fy0_row = df[(df['borrower_id']==BID) & (df['period']=='FY0')].iloc[0]
zscore_detail = zt.loc['FY0'].to_dict()

narrative = generate_borrower_commentary(
    borrower_name=borrower_name,
    industry=industry,
    revenue_fy0=fy0_row['revenue'],
    scorecard_result=sc,
    trends=trends,
    wc_detail=wc,
    merton=merton,
    zscore_detail=zscore_detail,
    pricing=pricing,
)

print(f'\n{"="*70}')
print(f'SECTION 8: CREDIT COMMITTEE NARRATIVE')
print(f'{"="*70}\n')
print(narrative)

## 9. Automated Comments Table (Consolidated)

In [ ]:
comments_table = generate_automated_comments_table(
    scorecard_result=sc,
    trends=trends,
    wc_detail=wc,
    merton=merton,
    zscore_detail=zscore_detail,
    pricing=pricing,
)

print(f'\n{"="*70}')
print(f'AUTOMATED COMMENTS TABLE (maps to Automated_Comments sheet)')
print(f'{"="*70}')

def colour_flag(val):
    if val in ('POSITIVE', 'GREEN'): return 'background-color: #C8E6C9'
    elif val in ('NEGATIVE', 'RED'): return 'background-color: #FFCDD2'
    return 'background-color: #FFF9C4'

display(
    comments_table.style.applymap(colour_flag, subset=['status'])
)

## 10. Export Results

In [ ]:
# Save key outputs
sc['scorecard_table'].to_csv('../outputs/tables/scorecard_reference_borrower.csv', index=False)
comments_table.to_csv('../outputs/tables/automated_comments_reference.csv', index=False)
trends.to_csv('../outputs/tables/trend_analysis_reference.csv', index=False)

# Save narrative as text
with open('../outputs/tables/credit_narrative_reference.txt', 'w') as f:
    f.write(f'CREDIT PAPER — {borrower_name}\n')
    f.write(f'Date: FY0 Assessment\n')
    f.write(f'Analyst: Automated\n')
    f.write(f'{"="*70}\n\n')
    f.write(narrative)

print('Outputs saved to outputs/tables/')
print('  - scorecard_reference_borrower.csv')
print('  - automated_comments_reference.csv')
print('  - trend_analysis_reference.csv')
print('  - credit_narrative_reference.txt')

---

## Summary: What This Demonstrates

This notebook shows the complete credit analyst workflow:

| Step | Output | Bank Use |
|------|--------|----------|
| Spreading | Standardised 3-year FS | Foundation for all analysis |
| Normalisation | EBITDAO | True operating cash generation |
| Ratio Analysis | 20+ ratios with trends | Quantitative assessment |
| Working Capital | Quality flag (GREEN/AMBER/RED) | Liquidity risk |
| Z-Score | Default zone + trend | Quick health check |
| Merton PD | PD, PVEL, implied LGD | IRB capital, pricing |
| Scorecard | Grade (A/B/C/D) + decision | Committee recommendation |
| Pricing | All-in rate build-up | Commercial negotiation |
| Covenants | Recommended financial covenants | Ongoing monitoring |
| Narrative | Auto-generated commentary | Credit paper documentation |

**This is the day-to-day work of a commercial credit analyst in an Australian bank.**